<a href="https://colab.research.google.com/github/24p11/recode-icd/blob/main/postprocess_mistral_report_generation_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import json
import ast
import re
import os
from utils  import *

In [2]:
file_name_10000 = "crh_sample_10000_20250924.csv"
file_name_5000 = "crh_sample_5000_20250919.csv"

In [3]:
df_gen_reports1 = pd.read_csv("data/" + file_name_10000, index_col=0).reset_index(drop=True)
df_gen_reports1.shape

(9999, 46)

In [4]:
df_gen_reports2 = pd.read_csv("data/" + file_name_5000, index_col=0).reset_index(drop=True)
df_gen_reports2.shape

(5000, 45)

In [5]:
df_gen_reports = pd.concat([df_gen_reports1,df_gen_reports2]).reset_index()
df_gen_reports.shape

(14999, 48)

In [6]:
df_gen_reports.columns

Index(['index', 'age', 'cage', 'cage2', 'sexe', 'date_entry', 'date_discharge',
       'date_of_birth', 'first_name', 'last_name', 'icd_primary_code',
       'icd_primary_description', 'icd_parent_code', 'case_management_type',
       'case_management_type_description', 'coding_rule',
       'case_management_type_text', 'drg_parent_code',
       'drg_parent_description', 'icd_secondaray_code',
       'text_secondary_icd_official', 'procedure', 'text_procedure',
       'admission_type', 'admission_mode', 'discharge_disposition', 'dms',
       'los_mean', 'los_sd', 'cancer_stage', 'score_TNM', 'histological_type',
       'treatment_recommandation', 'chemotherapy_regimen', 'biomarkers',
       'first_name_med', 'last_name_med', 'department', 'hospital',
       'template_name', 'user_prompt', 'system_prompt', 'prefix', 'prefix_len',
       'bacth', 'num_in_', 'crh', 'cd_md_pec'],
      dtype='object')

In [7]:
df_gen_reports["clinical_note"] = ""
df_gen_reports["annot_diagnostics"] = ""
df_gen_reports["annot_diagnostics_v2"] = ""
df_gen_reports["annot_criteria"] = ""

for i in range(len(df_gen_reports)):
  response = df_gen_reports.iloc[i]["crh"]
  clinical_note, annot_diag, annot_criteria = extract_generations_annotations(response)
  df_gen_reports.at[i, "clinical_note"] = clinical_note
  df_gen_reports.at[i, "annot_diagnostics"] = annot_diag
  df_gen_reports.at[i, "annot_diagnostics_v2"] = change_codes_target_format(annot_diag)
  df_gen_reports.at[i, "annot_criteria"] = annot_criteria
  df_gen_reports.at[i, "procedure_json"] =  json.dumps([df_gen_reports.iloc[i]["procedure"]])

In [8]:
import numpy as np

shuffled_indices = np.random.permutation(len(df_gen_reports))

# Réorganiser le dataframe selon les indices mélangés
df_gen_reports = df_gen_reports.iloc[shuffled_indices].reset_index(drop=True)

# Mélanger les indices
indices = np.arange(len(df_gen_reports))
np.random.shuffle(indices)

# Calculer les tailles des ensembles
train_size = int(len(df_gen_reports) * 0.7)
test_size = int(len(df_gen_reports) * 0.15)

# Scinder les indices
train_indices = indices[:train_size]
test_indices = indices[train_size:train_size + test_size]
valid_indices = indices[train_size + test_size:]

# Créer les dataframes
df_train = df_gen_reports.iloc[train_indices]
df_test = df_gen_reports.iloc[test_indices]
df_valid = df_gen_reports.iloc[valid_indices]

In [9]:
cols1 = ["index","department","clinical_note","procedure","icd_primary_code"]
cols2 = ["id","specialite","texte","code_acte_ccam","code_cim_dp"]
mapping = dict(zip(cols1, cols2))
path_out = "C:/data/wd/"
df_train[cols1].rename(mapping).to_csv(path_out +"train_format1.csv",index=False)
df_test[cols1].rename(mapping).to_csv(path_out +"test_format1.csv",index=False)
df_valid[cols1].rename(mapping).to_csv(path_out +"valid_format1.csv",index=False)

In [29]:
cols1 = ["index","clinical_note","procedure_json","annot_diagnostics_v2"]
cols2 = ["id","note","procedures_codes","icd_codes"]
mapping = dict(zip(cols1, cols2))
path_out = "C:/data/wd/"
df_train[cols1].rename(mapping).to_csv(path_out +"train_format2.csv",index=False)
df_test[cols1].rename(mapping).to_csv(path_out +"test_format2.csv",index=False)
df_valid[cols1].rename(mapping).to_csv(path_out +"valid_format2.csv",index=False)

# Ambulatory surgery corpa

In [15]:
df_ghm_chir_ambu = pd.read_excel("data/liste_ghm_chirurgie_ambulatoire_20251213.xlsx").reset_index(drop=True)

In [21]:
racine_chir_ambu = df_ghm_chir_ambu.ghm.str.slice(0,5).to_list()

In [34]:
# Selection of Outpatient and surgery (os)
df_gen_reports_os = df_gen_reports[ (df_gen_reports["admission_type"] =="Outpatient") & (df_gen_reports["drg_parent_code"].isin(racine_chir_ambu))].copy().reset_index()

In [35]:
df_gen_reports_os["clinical_note"] = ""
df_gen_reports_os["annot_diagnostics"] = ""
df_gen_reports_os["annot_diagnostics_v2"] = ""
df_gen_reports_os["annot_criteria"] = ""

for i in range(len(df_gen_reports_os)):
  response = df_gen_reports_os.iloc[i]["crh"]
 
  clinical_note, annot_diag, annot_criteria = extract_generations_annotations(response)
  df_gen_reports_os.at[i, "clinical_note"] = clinical_note
  df_gen_reports_os.at[i, "annot_diagnostics"] = annot_diag
  df_gen_reports_os.at[i, "annot_diagnostics_v2"] = change_codes_target_format(annot_diag)
  df_gen_reports_os.at[i, "annot_criteria"] = annot_criteria
  df_gen_reports_os.at[i, "procedure_json"] =  json.dumps([df_gen_reports_os.iloc[i]["procedure"]])

In [38]:
shuffled_indices = np.random.permutation(len(df_gen_reports_os))

# Réorganiser le dataframe selon les indices mélangés
df_gen_reports_os = df_gen_reports_os.iloc[shuffled_indices].reset_index(drop=True)

# Mélanger les indices
indices = np.arange(len(df_gen_reports_os))
np.random.shuffle(indices)

# Calculer les tailles des ensembles
train_size = int(len(df_gen_reports_os) * 0.7)
test_size = int(len(df_gen_reports_os) * 0.15)

# Scinder les indices
train_indices = indices[:train_size]
test_indices = indices[train_size:train_size + test_size]
valid_indices = indices[train_size + test_size:]

# Créer les dataframes
df_train = df_gen_reports_os.iloc[train_indices]
df_test = df_gen_reports_os.iloc[test_indices]
df_valid = df_gen_reports_os.iloc[valid_indices]

In [39]:
cols1 = ["index","department","clinical_note","procedure","icd_primary_code"]
cols2 = ["id","specialite","texte","code_acte_ccam","code_cim_dp"]
mapping = dict(zip(cols1, cols2))
path_out = "C:/data/wd/"
df_train[cols1].rename(mapping).to_csv(path_out +"train_chir_ambu_format1.csv",index=False)
df_test[cols1].rename(mapping).to_csv(path_out +"test_chir_ambu_format1.csv",index=False)
df_valid[cols1].rename(mapping).to_csv(path_out +"valid_chir_ambu_format1.csv",index=False)

In [40]:
cols1 = ["index","clinical_note","procedure_json","annot_diagnostics_v2"]
cols2 = ["id","note","procedures_codes","icd_codes"]
mapping = dict(zip(cols1, cols2))
path_out = "C:/data/wd/"
df_train[cols1].rename(mapping).to_csv(path_out +"train_chir_ambu_format2.csv",index=False)
df_test[cols1].rename(mapping).to_csv(path_out +"test_chir_ambu_format2.csv",index=False)
df_valid[cols1].rename(mapping).to_csv(path_out +"valid_chir_ambu_format2.csv",index=False)